# Concept reanalysis from cached representations
Run after `experiments/final_experiment.ipynb`; no GPU is required.

In [1]:
from pathlib import Path

# Resolve the project from either its root or a notebook's folder.
current_directory = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in [current_directory, *current_directory.parents]
    if (path / "requirements.txt").is_file() and (path / "data_preparation").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Keep the three project folders and requirements.txt together.")
DATA_DIR = PROJECT_ROOT / "data_preparation"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments"
METRICS_DIR = PROJECT_ROOT / "metrics_export"

import numpy as np
import pandas as pd
from IPython.display import display

if not (DATA_DIR / "random_concept_pairs.csv").exists():
    get_ipython().run_line_magic("run", str(DATA_DIR / "cached_data_preparation.ipynb"))
pairs = pd.read_csv(DATA_DIR / "random_concept_pairs.csv")
rows = []
frozen_values = {}
frozen_bootstrap = {}
distance_rows = []
frozen_distances = {}
frozen_distance_bootstrap = {}

# Independent class draws within each dataset, shared across conditions/models.
rng = np.random.default_rng(15)
def class_draws():
    return np.concatenate([
        rng.choice(np.flatnonzero(pairs.dataset.eq(dataset)), size=(1000, int(pairs.dataset.eq(dataset).sum())), replace=True)
        for dataset in ["deepset", "safeguard"]], axis=1)
injection_draws = class_draws()
benign_draws = class_draws()

def cosine(first, second):
    denominator = np.linalg.norm(first, axis=-1) * np.linalg.norm(second, axis=-1)
    return np.where(denominator > 1e-8,
        (first * second).sum(axis=-1) / np.maximum(denominator, 1e-8), np.nan)

def concept_distances(first, second):
    first_norm = np.linalg.norm(first, axis=-1)
    second_norm = np.linalg.norm(second, axis=-1)
    raw = np.linalg.norm(second - first, axis=-1)
    first_unit = first / np.maximum(first_norm[..., None], 1e-8)
    second_unit = second / np.maximum(second_norm[..., None], 1e-8)
    return {"euclidean": raw,
            "relative_euclidean": raw / np.maximum(first_norm, 1e-8),
            "unit_euclidean": np.linalg.norm(second_unit - first_unit, axis=-1)}

def vectors(frame, draws=None):
    injections = frame.loc[pairs.injection_row_id].to_numpy(dtype=np.float64)
    benign = frame.loc[pairs.benign_row_id].to_numpy(dtype=np.float64)
    if draws is None:
        return injections.mean(axis=0) - benign.mean(axis=0)
    return injections[draws[0]].mean(axis=1) - benign[draws[1]].mean(axis=1)

for method, seed in [("frozen", 15), ("lora", 15), ("lora", 16), ("lora", 17)]:
    with np.load(EXPERIMENT_DIR / f"representations_{method}_{seed}.npz", allow_pickle=False) as cache:
        keys = pd.DataFrame({"row_id": cache["variant_row_ids"], "column": cache["variant_columns"]})
        features = cache["variant_embeddings"]
        conditions = {}
        for style in ["clean", "zero_width", "homoglyph", "base64"]:
            mask = keys["column"].str.endswith("_" + style)
            conditions[style] = pd.DataFrame(features[mask], index=keys.loc[mask, "row_id"])
        clean = vectors(conditions["clean"])
        clean_bootstrap = vectors(conditions["clean"], (injection_draws, benign_draws))
        for style, frame in conditions.items():
            changed = vectors(frame)
            changed_bootstrap = vectors(frame, (injection_draws, benign_draws))
            distances = concept_distances(clean, changed)
            bootstrap_distances = concept_distances(clean_bootstrap, changed_bootstrap)
            for metric, estimate in distances.items():
                key = (style, metric)
                samples = bootstrap_distances[metric]
                if method == "frozen":
                    frozen_distances[key] = float(estimate)
                    frozen_distance_bootstrap[key] = samples
                lower_distance, upper_distance = np.percentile(samples, [2.5, 97.5])
                distance_change = float(estimate) - frozen_distances[key]
                change_samples = samples - frozen_distance_bootstrap[key]
                lower_change, upper_change = np.percentile(change_samples, [2.5, 97.5])
                distance_rows.append({"value": method, "seed": seed, "style": style,
                    "pairs": len(pairs), "metric": metric, "distance": float(estimate),
                    "lower": lower_distance, "upper": upper_distance,
                    "change_vs_frozen": distance_change,
                    "change_lower": lower_change, "change_upper": upper_change})
            similarity = float(cosine(clean, changed))
            bootstrap = cosine(clean_bootstrap, changed_bootstrap)
            if method == "frozen":
                frozen_values[style] = similarity
                frozen_bootstrap[style] = bootstrap
            lower, upper = np.nanpercentile(bootstrap, [2.5, 97.5])
            difference_bootstrap = bootstrap - frozen_bootstrap[style]
            change_lower, change_upper = np.nanpercentile(difference_bootstrap, [2.5, 97.5])
            rows.append({"value": method, "seed": seed, "style": style, "pairs": len(pairs),
                "R": similarity, "lower": lower, "upper": upper,
                "change_vs_frozen": similarity - frozen_values[style],
                "change_lower": change_lower, "change_upper": change_upper,
                "clean_concept_norm": np.linalg.norm(clean),
                "perturbed_concept_norm": np.linalg.norm(changed),
                "undefined_bootstrap_samples": int(np.isnan(bootstrap).sum())})
results = pd.DataFrame(rows)
results.to_csv(METRICS_DIR / "random_concept_robustness.csv", index=False)
display(results[results["style"].ne("clean")])

# Lower distance indicates greater clean/perturbed similarity; raw values include scale.
distance_results = pd.DataFrame(distance_rows)
distance_results.to_csv(METRICS_DIR / "random_concept_distances.csv", index=False)
display(distance_results[distance_results["style"].ne("clean")])
unit_results = distance_results[distance_results.metric.eq("unit_euclidean")].reset_index(drop=True)
assert np.allclose(unit_results.distance, np.sqrt(np.maximum(0, 2 - 2 * results.R)), atol=1e-7)


Prepared cached-analysis inputs: 1640 source records and 20 pairs


,value,seed,style,pairs,R,lower,upper,change_vs_frozen,change_lower,change_upper,clean_concept_norm,perturbed_concept_norm,undefined_bootstrap_samples
1,frozen,15,zero_width,20,-0.066544,-0.278939,0.309760,0.000000,0.000000,0.000000,4.257671,2.081187,0
2,frozen,15,homoglyph,20,0.221331,-0.246805,0.537443,0.000000,0.000000,0.000000,4.257671,2.659109,0
3,frozen,15,base64,20,0.029371,-0.320500,0.353428,0.000000,0.000000,0.000000,4.257671,1.381115,0
5,lora,15,zero_width,20,-0.026856,-0.286045,0.249403,0.039688,-0.285367,0.298672,7.217134,2.264600,0
6,lora,15,homoglyph,20,0.091142,-0.229886,0.450936,-0.130188,-0.289894,0.199351,7.217134,2.639695,0
7,lora,15,base64,20,-0.040756,-0.333218,0.222518,-0.070127,-0.307656,0.281397,7.217134,1.407678,0
9,lora,16,zero_width,20,-0.042934,-0.339122,0.260476,0.023610,-0.351157,0.326899,7.997925,2.214679,0
10,lora,16,homoglyph,20,0.160290,-0.219964,0.550676,-0.061041,-0.321790,0.310749,7.997925,2.751149,0
11,lora,16,base64,20,-0.013282,-0.366207,0.274405,-0.042653,-0.373565,0.372698,7.997925,1.433291,0
13,lora,17,zero_width,20,-0.032117,-0.305711,0.252507,0.034427,-0.350385,0.322897,8.885750,2.237663,0


,value,seed,style,pairs,metric,distance,lower,upper,change_vs_frozen,change_lower,change_upper
3,frozen,15,zero_width,20,euclidean,4.861934,4.479087,8.502745,0.000000,0.000000,0.000000
4,frozen,15,zero_width,20,relative_euclidean,1.141923,0.966880,1.381868,0.000000,0.000000,0.000000
5,frozen,15,zero_width,20,unit_euclidean,1.460510,1.174938,1.599337,0.000000,0.000000,0.000000
6,frozen,15,homoglyph,20,euclidean,4.492993,3.992043,8.882959,0.000000,0.000000,0.000000
7,frozen,15,homoglyph,20,relative_euclidean,1.055270,0.855017,1.359010,0.000000,0.000000,0.000000
8,frozen,15,homoglyph,20,unit_euclidean,1.247934,0.961829,1.579117,0.000000,0.000000,0.000000
9,frozen,15,base64,20,euclidean,4.437321,4.239930,8.257452,0.000000,0.000000,0.000000
10,frozen,15,base64,20,relative_euclidean,1.042194,0.942950,1.209551,0.000000,0.000000,0.000000
11,frozen,15,base64,20,unit_euclidean,1.393290,1.137164,1.625115,0.000000,0.000000,0.000000
15,lora,15,zero_width,20,euclidean,7.621897,6.600322,11.240267,2.759963,0.686244,3.996988
